<a href="https://colab.research.google.com/github/umarrashid952/Python_basics/blob/master/Test_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
torch.cuda.is_available()

True

### 1. Logistic regression forward pass

In [3]:
import torch.nn.functional as F

y = torch.tensor([1.0])   # True label
x = torch.tensor([1.1])   # Input features
w = torch.tensor([2.2])   # Weight parameter
b = torch.tensor([0.0])   # Bias unit
z = w * x + b             # Net input
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(y, a)
print(loss)

tensor(8.1660)


### 2. Computing gradients via autograd

In [4]:
import torch.nn.functional as F
from torch.autograd import grad

y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2], requires_grad= True)
b = torch.tensor([0.0], requires_grad= True)

z = x1 * w1 + b
a = torch.sigmoid(z)

loss = F.binary_cross_entropy(a, y)

grad_L_w1 = grad(loss, w1, retain_graph=True)
grad_L_b = grad(loss, b, retain_graph=True)

print(grad_L_w1)
print(grad_L_b)

(tensor([-0.0898]),)
(tensor([-0.0817]),)


### 3. A multilayer perceptron with two hidden layers

In [19]:
class NeuralNetwork(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()

        self.layers = torch.nn.Sequential(
            # 1st hidden layer
            torch.nn.Linear(num_inputs, 30),
            torch.nn.ReLU(),

            # 2nd hidden layer
            torch.nn.Linear(30, 20),
            torch.nn.ReLU(),

            #. Output layer
            torch.nn.Linear(20, num_outputs),
        )

    def forward(self, x):
        logits = self.layers(x)
        return logits



Instantiate a new neural network object

In [20]:
model = NeuralNetwork(50, 3)
print(model)

NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)


Calculate the total number of trainable parameters

In [21]:
num_param = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Total no of trainable parameters:", num_param)

Total no of trainable parameters: 2213


#### Weight parameter matrix of layer $0$

In [22]:
print(model.layers[0].weight)

Parameter containing:
tensor([[ 0.0340, -0.0099,  0.0833,  ..., -0.0590,  0.0906,  0.0944],
        [-0.0074, -0.0493,  0.1094,  ...,  0.0876,  0.0331,  0.1236],
        [ 0.1178,  0.0777,  0.0982,  ...,  0.0988, -0.1335, -0.1305],
        ...,
        [-0.0856,  0.0205,  0.0347,  ..., -0.0717, -0.0972,  0.0246],
        [ 0.1137, -0.0545, -0.0439,  ...,  0.0593,  0.1116,  0.0427],
        [ 0.0944, -0.0321,  0.0219,  ...,  0.1058, -0.1375,  0.0757]],
       requires_grad=True)


In [23]:
print(model.layers[0].weight.shape)

torch.Size([30, 50])


### Creating a small toy dataset

In [24]:
X_train = torch.tensor([
        [-1.2, 3.1],
 [-0.9, 2.9],
 [-0.5, 2.6],
 [2.3, -1.1],
 [2.7, -1.5]
])

y_train = torch.tensor([0, 0, 0, 1, 1])

X_test = torch.tensor([
    [-0.8, 2.8],
 [2.6, -1.6]
])
y_test = torch.tensor([0, 1])

### 5.Define a custom 'Dataset' class

In [25]:
# Instructions for retrieving one item at a time and the corresponding label
from torch.utils.data import Dataset

class ToyDataset(Dataset):
    def __init__(self, X, y):
        self.features = X
        self.labels = y

    def __getitem__(self, index):
        one_x = self.features[index]
        one_y = self.labels[index]
        return one_x, one_y

    def __len__(self):
      return self.labels.shape[0]

Train_ds = ToyDataset(X_train, y_train)
Test_ds = ToyDataset(X_test, y_test)

In [26]:
print(len(Train_ds))

5


### 6.  Instantiating data loaders

In [27]:
from torch.utils.data import DataLoader

torch.manual_seed(123)

# The ToyDataset instance created earlier serves as an input to the data loader
train_loader = DataLoader(dataset=Train_ds, batch_size=2, shuffle=True, num_workers=0)

test_loader = DataLoader(dataset=Test_ds, batch_size=2, shuffle=False, num_workers=0)



### Iterating over training dataset



In [28]:
for idx, (x, y) in enumerate(train_loader) :
  print(f"Batch {idx +1}:", x, y)

Batch 1: tensor([[ 2.3000, -1.1000],
        [-0.9000,  2.9000]]) tensor([1, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [-0.5000,  2.6000]]) tensor([0, 0])
Batch 3: tensor([[ 2.7000, -1.5000]]) tensor([1])


Iterating over test dataset

In [29]:
for idx, (x, y) in enumerate(test_loader):
  print(f"Batch {idx+1}:", x, y)


Batch 1: tensor([[-0.8000,  2.8000],
        [ 2.6000, -1.6000]]) tensor([0, 1])


### A training loader that drops the last batch

In [30]:
train_loader = DataLoader(dataset=Train_ds, batch_size=2, shuffle=True, num_workers=0, drop_last=True)

In [31]:
for idx, (x, y) in enumerate(train_loader) :
  print(f"Batch {idx +1}:", x, y)

Batch 1: tensor([[-0.5000,  2.6000],
        [-0.9000,  2.9000]]) tensor([0, 0])
Batch 2: tensor([[-1.2000,  3.1000],
        [ 2.3000, -1.1000]]) tensor([0, 1])


Neural network training in PyTorch

In [33]:
import torch.nn.functional as F

torch.manual_seed(123)

model = NeuralNetwork(num_inputs=2, num_outputs=2)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

num_epochs = 3
for epoch in range(num_epochs):
  model.train()

  for batch_idx, (features, labels) in enumerate(train_loader):
      logits = model(features)

      loss = F.cross_entropy(logits, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      # LOGGING
      print(f"Epoch {epoch+1:03d}/{num_epochs:03d}"
      f" | Batch {batch_idx:03d}/{len(train_loader)}"
      f" Train loss: {loss:.2f}")

model.eval()

Epoch 001/003 | Batch 000/2 Train loss: 0.75
Epoch 001/003 | Batch 001/2 Train loss: 0.65
Epoch 002/003 | Batch 000/2 Train loss: 0.44
Epoch 002/003 | Batch 001/2 Train loss: 0.13
Epoch 003/003 | Batch 000/2 Train loss: 0.03
Epoch 003/003 | Batch 001/2 Train loss: 0.00


NeuralNetwork(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=2, bias=True)
  )
)

After we have trained the model, we can use it to make predictions:

In [34]:
model.eval()

with torch.no_grad():
 outputs = model(X_train)
print(outputs)

tensor([[ 2.8569, -4.1618],
        [ 2.5382, -3.7548],
        [ 2.0944, -3.1820],
        [-1.4814,  1.4816],
        [-1.7176,  1.7342]])


In [36]:
torch.set_printoptions(sci_mode=False)
probas = torch.softmax(outputs, dim=1)
print(probas)

tensor([[0.9991, 0.0009],
        [0.9982, 0.0018],
        [0.9949, 0.0051],
        [0.0491, 0.9509],
        [0.0307, 0.9693]])


In [37]:
predictions = torch.argmax(probas, dim=1)
print(predictions)

tensor([0, 0, 0, 1, 1])


In [38]:
predictions == y_train

tensor([True, True, True, True, True])